# Day 060 — Exercise 3: Cached API

Wire `SimpleCache` and `cache_aside` into a FastAPI app. The `POST /ask` endpoint checks the cache before calling the (slow) process function. The `cache_hit` flag in the response tells the client whether a fresh answer was generated or a cached one was returned.

`GET /cache/stats` gives operational visibility — essential for tuning your TTL. A low hit rate means TTL is too short or prompts vary too much.

In [ ]:
# --- Provided: SimpleCache (from Exercise 1) ---
import time
from typing import Any

class SimpleCache:
    def __init__(self):
        self._store: dict = {}

    def set(self, key: str, value: Any, ttl: float = 60.0) -> None:
        self._store[key] = (value, time.monotonic() + ttl)

    def get(self, key: str) -> Any:
        entry = self._store.get(key)
        if entry is None:
            return None
        value, expires_at = entry
        if time.monotonic() > expires_at:
            del self._store[key]
            return None
        return value

    def has(self, key: str) -> bool:
        return self.get(key) is not None

    def clear(self) -> int:
        n = len(self._store)
        self._store.clear()
        return n

    def __len__(self) -> int:
        now = time.monotonic()
        return sum(1 for _, exp in self._store.values() if now <= exp)


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel, Field
from starlette.testclient import TestClient


## Task

Implement `build_cached_api(process_fn=None)` with:

```
POST /ask    {"prompt": "..."}  → {"answer": str, "cache_hit": bool}
GET /cache/stats               → {"hits": int, "misses": int, "size": int}
DELETE /cache                  → {"cleared": int}
```

- Cache by `prompt` string; use `cache.get(prompt)` / `cache.set(prompt, answer)`
- Track `hits` and `misses` in a dict
- `DELETE /cache` resets both the cache and the hit/miss counters

## Your Implementation

In [ ]:
def build_cached_api(process_fn=None) -> FastAPI:
    """FastAPI with response caching.

    POST /ask    {"prompt": "..."}  → {"answer": str, "cache_hit": bool}
    GET /cache/stats               → {"hits": int, "misses": int, "size": int}
    DELETE /cache                  → {"cleared": int}

    process_fn: optional callable(prompt: str) -> str for testing.
    Returns 422 when prompt is empty.
    """
    # TODO: create SimpleCache, stats dict, POST /ask, GET /cache/stats, DELETE /cache
    raise NotImplementedError


In [ ]:
def build_cached_api(process_fn=None) -> FastAPI:
    app   = FastAPI()
    cache = SimpleCache()
    stats = {"hits": 0, "misses": 0}

    class _AskReq(BaseModel):
        prompt: str = Field(min_length=1)

    @app.post("/ask")
    def ask(req: _AskReq):
        cached = cache.get(req.prompt)
        if cached is not None:
            stats["hits"] += 1
            return {"answer": cached, "cache_hit": True}
        stats["misses"] += 1
        answer = process_fn(req.prompt) if process_fn else req.prompt.upper()
        cache.set(req.prompt, answer, ttl=60.0)
        return {"answer": answer, "cache_hit": False}

    @app.get("/cache/stats")
    def cache_stats():
        return {"hits": stats["hits"], "misses": stats["misses"], "size": len(cache)}

    @app.delete("/cache")
    def clear_cache():
        n = cache.clear()
        stats["hits"] = 0
        stats["misses"] = 0
        return {"cleared": n}

    return app


## Automated checks

In [ ]:
score, total = 0, 6
try:
    app    = build_cached_api(process_fn=str.upper)
    client = TestClient(app, raise_server_exceptions=False)

    # first ask — miss
    r1 = client.post("/ask", json={"prompt": "hello"})
    assert r1.status_code == 200, f"Got {r1.status_code}"
    b1 = r1.json()
    assert b1["cache_hit"] is False, f"Expected cache_hit=False: {b1}"
    assert b1["answer"] == "HELLO"
    score += 1; print("\u2705 first POST /ask returns correct answer, cache_hit=False")

    # second ask — hit
    r2 = client.post("/ask", json={"prompt": "hello"})
    b2 = r2.json()
    assert b2["cache_hit"] is True, f"Expected cache_hit=True: {b2}"
    assert b2["answer"] == "HELLO"
    score += 1; print("\u2705 repeated prompt returns cache_hit=True")

    # different prompt — separate cache entry
    r3 = client.post("/ask", json={"prompt": "world"})
    assert r3.json()["cache_hit"] is False
    score += 1; print("\u2705 different prompt is a separate cache entry")

    # stats
    rs = client.get("/cache/stats")
    s = rs.json()
    assert s["hits"] == 1, f"Expected 1 hit, got {s['hits']}"
    assert s["misses"] == 2, f"Expected 2 misses, got {s['misses']}"
    assert s["size"] == 2, f"Expected size 2, got {s['size']}"
    score += 1; print("\u2705 /cache/stats reports correct hits/misses/size")

    # clear cache
    rd = client.delete("/cache")
    assert rd.json()["cleared"] == 2, f"Expected cleared=2, got {rd.json()}"
    rs2 = client.get("/cache/stats")
    assert rs2.json()["size"] == 0
    score += 1; print("\u2705 DELETE /cache clears all entries")

    # empty prompt → 422
    r4 = client.post("/ask", json={"prompt": ""})
    assert r4.status_code == 422
    score += 1; print("\u2705 empty prompt → 422")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def build_cached_api(process_fn=None) -> FastAPI:
    app   = FastAPI()
    cache = SimpleCache()
    stats = {"hits": 0, "misses": 0}

    class _AskReq(BaseModel):
        prompt: str = Field(min_length=1)

    @app.post("/ask")
    def ask(req: _AskReq):
        cached = cache.get(req.prompt)
        if cached is not None:
            stats["hits"] += 1
            return {"answer": cached, "cache_hit": True}
        stats["misses"] += 1
        answer = process_fn(req.prompt) if process_fn else req.prompt.upper()
        cache.set(req.prompt, answer, ttl=60.0)
        return {"answer": answer, "cache_hit": False}

    @app.get("/cache/stats")
    def cache_stats():
        return {"hits": stats["hits"], "misses": stats["misses"], "size": len(cache)}

    @app.delete("/cache")
    def clear_cache():
        n = cache.clear()
        stats["hits"] = 0
        stats["misses"] = 0
        return {"cleared": n}

    return app
```

**Cache key choice:** using the raw `prompt` string as the key means two semantically identical prompts with different whitespace are treated as misses. In production you might normalise the key: `prompt.strip().lower()`. For Ollama responses, case and whitespace in the prompt usually DO produce different answers, so the raw string is a reasonable default.

</details>